# Task vs progress: what does the pivot read the goal from?

The KV-transfer results argue for **repeated re-reading** of the goal from the task span, against a **sentinel** view
(a goal marker built up in the progress tokens and carried forward). This notebook fills the 2 × 2 grid of *which span the
pivot reads from the source run*, on the multi-predicate setup:

```
p_src: a'b'c -> c     source (donor)          p_dst: a'b'd -> d     the patched run (prompt ends at "{var} =")
```

| | progress from DST | progress from SRC |
|---|---|---|
| **task from DST** | no-op (clean DST) | pre-commitment swap: 6 % (residual, all layers) |
| **task from SRC** | task-span KV, goal-250: 37 % (V×1), 80 % (V×4) | ≈ 99 % |

Spans: **task** = the task list in the user turn (`\n`-padded so SRC/DST align); **progress** = `<|im_end|> … {var}` (the
prefilled code; token-identical in SRC and DST). The pivot ` =` is never overwritten; everything is applied **from the
pivot onward**.

**Methods**
* **`kv`** (`PP.intervention_stack`, I3 only): the chosen heads (`goal250` or `all`) read SRC's k/v on the span from the
  pivot on; values × `V`. Every other key, and every pre-pivot latent, stays as in the clean DST run.
* **`residual`**: every layer's output residual on the span ← SRC's; every *other* position between the span and the
  pivot ← clean DST's, so it stays identical to the clean DST run (e.g. task swapped ⇒ the progress is pinned to DST).

At V×1, `kv` with **all** heads and `residual` give the pivot the same keys and values, so they should agree (checked
below). The open question: does boosting the *progress* values (V×4) raise the 6 %? That would mean the progress carries
a pre-commitment that contributes, though less than the task span.

Same instance sampler as the batch runs (`PP.sample_multi_instances`, seed 1123): instance *i* here is instance *i* in
`path_patch_eval.py` / `precommit_swap_eval.py`. Python → Python for now.

In [1]:
%load_ext autoreload
%autoreload 2
import os, sys

os.environ.setdefault("CUDA_VISIBLE_DEVICES", "4")   # check nvidia-smi first (shared machine)
os.environ["TOKENIZERS_PARALLELISM"] = "false"

sys.path.append("../..")  # repo root, so `src` is importable
import json, collections
from contextlib import ExitStack
from types import SimpleNamespace
import numpy as np
import torch
import src.diff.functional as P
import src.diff.functional as FUNC
from src.diff.functional import *          # helpers into scope
import src.diff.path_patch as PP            # geometry / k-v routing / residual tracer + patcher / oracle (batch eval's code)

P.setup("Qwen/Qwen2.5-Coder-32B-Instruct") # loads model + head masks (once)
mt = P.mt
with open(os.path.join(os.path.dirname(env_utils.DEFAULT_RESULTS_DIR),
                       "head_sets/Qwen2.5-Coder-32B-Instruct/goal_topk/goal__lam20_top250.json")) as f:
    GOAL_HEADS_250 = [(l, h) for l, h in json.load(f)]
ALL_HEADS = [(l, h) for l in range(mt.n_layer) for h in range(mt.n_heads)]
from rich.panel import Panel
from rich.table import Table
from rich.text import Text
from rich.console import Console
from rich.markup import escape
print(f"{len(GOAL_HEADS_250)} goal heads · {len(ALL_HEADS)} heads in total")

2026-10-21 23:24:43 src.models WARNING  Qwen/Qwen2.5-Coder-32B-Instruct not found in /disk/u/arnab/Codes/Models
If not found in cache, model will be downloaded from HuggingFace to cache directory: /disk/u/models


Loading weights:   0%|          | 0/771 [00:00<?, ?it/s]

Qwen/Qwen2.5-Coder-32B-Instruct
n_heads=40  n_kv_heads=8  group=5  head_dim=128  n_layer=64
GOAL_HEADS     : 122  (layers 28–55)
BOUNDARY_HEADS : 140  (layers 11–62)
UNION_HEADS    : 228
250 goal heads · 2560 heads in total


## Knobs

In [2]:
#------------------------------------------------------------------------------------------------------
DATATYPE = "list[str]"            # int | str | list[int] | list[str]
SRC_LANG, DST_LANG = "python", "python"
SEED, N_SAMPLES = 1123, 32        # PP.sample_multi_instances (the batch eval's draw)
IDX = 0                           # instance for the single-instance cells
READOUT_HEADS = GOAL_HEADS_250
READOUT = True                    # GoalScope readout at ` =` in the single-instance run (the sweep has its own knob)
MAX_NEW = 64
ORACLE_MODEL = "claude-sonnet-4-6"
HEADSETS = {"goal250": GOAL_HEADS_250, "all": ALL_HEADS}
# (method, heads, span, V) — "none" = the clean DST run (no-op corner of the grid)
CONFIGS = [
    ("none", None, "none", 1.0),
    ("kv", "goal250", "task", 1.0), ("kv", "goal250", "task", 4.0),
    ("kv", "goal250", "progress", 1.0), ("kv", "goal250", "progress", 4.0),
    ("kv", "goal250", "task+progress", 1.0),
    ("kv", "all", "task", 1.0), ("kv", "all", "task", 4.0),
    ("kv", "all", "progress", 1.0), ("kv", "all", "progress", 4.0),
    ("kv", "all", "task+progress", 1.0),
    ("residual", None, "task", 1.0), ("residual", None, "progress", 1.0), ("residual", None, "task+progress", 1.0),
]
#------------------------------------------------------------------------------------------------------
PP_CTX = SimpleNamespace(mt=mt, group=P.GROUP, readout_heads=READOUT_HEADS, gsp=getattr(P, "gsp", None))
LAYERS = list(range(mt.n_layer))
cfg_name = lambda m, h, s, v: "clean DST (no-op)" if m == "none" else f"{m}{'·' + h if h else ''} · {s} · V×{v:g}"

## Geometry — task span, progress span, pivot

`PP.build_geometry(window="span", span="list")` gives the `\n`-padded task-list windows and the I2 progress window
(`<|im_end|> … pivot-1` on DST); the SRC progress positions are the same tokens aligned from the end (asserted). Tokens
between the task list and the progress (the rest of the user turn), if any, are shown in grey (task span: light red in SRC, light green in DST; progress: yellow; pivot: salmon): they are never transferred
(`kv`: read from DST; `residual`: pinned to DST).

In [3]:
def geometry(inst):
    g = PP.build_geometry(inst, window="span", span="list", ctx=PP_CTX)
    s_ids, d_ids = PP._ids(g["src_tok"]), PP._ids(g["dst_tok"])
    L_s, L_d = len(s_ids), len(d_ids)
    prog_d = list(g["seg"])                                   # <|im_end|> .. pivot-1 on DST
    prog_s = [q - L_d + L_s for q in prog_d]
    assert [s_ids[q] for q in prog_s] == [d_ids[q] for q in prog_d], "SRC/DST progress tokens differ"
    task_s, task_d = list(g["src_w"]), list(g["dst_w"])
    assert len(task_s) == len(task_d) and not set(task_d) & set(prog_d)
    W = {"none": ([], []), "task": (task_s, task_d), "progress": (prog_s, prog_d),
         "task+progress": (task_s + prog_s, task_d + prog_d)}
    gap = [q for q in range(max(task_d) + 1, min(prog_d))]
    return dict(inst=inst, g=g, src_tok=g["src_tok"], dst_tok=g["dst_tok"], W=W, gap=gap,
                piv_s=L_s - 1, piv_d=L_d - 1, L_s=L_s, L_d=L_d)


TASK_STYLE = {"src": "black on #ffb3b3", "dst": "black on #b8f0b8"}   # source task span: light red · destination: light green


def show_prompt(G, which="dst"):
    tok = G[f"{which}_tok"]; ids = PP._ids(tok)
    task = set(G["W"]["task"][0 if which == "src" else 1]); prog = set(G["W"]["progress"][0 if which == "src" else 1])
    gap = set(G["gap"]) if which == "dst" else set()
    piv = len(ids) - 1
    t = Text()
    for i, tid in enumerate(ids):
        s = mt.tokenizer.decode([tid]).replace("\n", "\\n\n")
        style = ("bold black on salmon1" if i == piv else TASK_STYLE[which] if i in task
                 else "black on yellow1" if i in prog else "grey50" if i in gap else "")
        t.append(s, style=style)
    tcol = "light red" if which == "src" else "light green"
    rich_print(Panel(t, title=f"[bold]{which.upper()}[/bold] · task span ({tcol}) · progress (yellow) · pivot (salmon) · "
                              f"L={len(ids)}", title_align="left", border_style="grey42"))


INSTS, _sample_seed = PP.sample_multi_instances(DATATYPE, SRC_LANG, DST_LANG, n_samples=N_SAMPLES, seed=SEED)
G = geometry(INSTS[IDX])
inst = G["inst"]
print(f"[{IDX}] SRC c = {inst.src_frontier.id}: {inst.src_frontier.description!r}\n"
      f"     DST d = {inst.dst_frontier.id}: {inst.dst_frontier.description!r}")
print(f"task {len(G['W']['task'][1])} tok · progress {len(G['W']['progress'][1])} tok · gap {len(G['gap'])} tok "
      f"· pad {G['g']['pad_src']}/{G['g']['pad_dst']} (src/dst) · pivot {mt.tokenizer.decode([PP._ids(G['dst_tok'])[-1]])!r}")
show_prompt(G, "src"); show_prompt(G, "dst")

[0] SRC c = count_long: 'the number of words longer than five characters'
     DST d = longest_word_length: 'the length of the longest word'
task 34 tok · progress 56 tok · gap 1 tok · pad 0/2 (src/dst) · pivot ' ='


╭─ SRC · task span (light red) · progress (yellow) · pivot (salmon) · L=155 ──────────────────────────────────────╮
│ <|im_start|>system\n                                                                                            │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>\n                                                            │
│ <|im_start|>user\n                                                                                              │
│ Write a Python program that implements a function `func` which, given a list of strings `words`, computes:\n    │
│ (1) the number of words that start with a vowel\n                                                               │
│ (2) whether any word is a palindrome\n                                                                          │
│ (3) the number of words longer than five characters\n                                                           │
│ <|im_end|>\n                                                                                                    │
│ <|im_start|>assistant\n                                                                                         │
│ <code>\n                                                                                                        │
│ def func(words: list[str]):\n                                                                                   │
│     a = sum(1 for w in words if w and w[0].lower() in "aeiou")\n                                                │
│     b = any(w == w[::-1] for w in words)\n                                                                      │
│     c =                                                                                                         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─ DST · task span (light green) · progress (yellow) · pivot (salmon) · L=155 ────────────────────────────────────╮
│ <|im_start|>system\n                                                                                            │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>\n                                                            │
│ <|im_start|>user\n                                                                                              │
│ Write a Python program that implements a function `func` which, given a list of strings `words`, computes:\n    │
│ \n                                                                                                              │
│ \n                                                                                                              │
│ (1) the number of words that start with a vowel\n                                                               │
│ (2) whether any word is a palindrome\n                                                                          │
│ (3) the length of the longest word\n                                                                            │
│ <|im_end|>\n                                                                                                    │
│ <|im_start|>assistant\n                                                                                         │
│ <code>\n                                                                                                        │
│ def func(words: list[str]):\n                                                                                   │
│     a = sum(1 for w in words if w and w[0].lower() in "aeiou")\n                                                │
│     b = any(w == w[::-1] for w in words)\n                                                                      │
│     c =                                                                                                         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## Interventions

`make_stack(G, method, span, heads, V)` → a zero-argument factory of a fresh context (single use per forward /
generation), so every config runs through the same `run_patched`.

`residual` also swaps the span's **input embeddings** (layer 0 reads k/v from the embeddings, not from a layer output), so at every layer the pivot sees exactly SRC's span: this is what makes it equal to `kv` with all heads at V×1.

In [4]:
from contextlib import contextmanager


def make_stack(G, method, span, heads=None, v_scale=1.0):
    src_pos, dst_pos = G["W"][span]
    if method == "none" or span == "none":
        return ExitStack
    if method == "kv":    # I3 only: the chosen heads read SRC's span k/v from the pivot on (values x v_scale)
        gk = dict(src_tok=G["src_tok"], dst_tok=G["dst_tok"], src_w=src_pos, dst_w=dst_pos)
        return PP.intervention_stack(gk, task_heads=[], goal_heads=HEADSETS[heads], i1=False, i2=False, i3=True,
                                     i3_query="last", v_scale=v_scale, alpha=1.0, ctx=PP_CTX)
    if method == "residual":   # span <- SRC; every other position between the span and the pivot <- clean DST
        assert v_scale == 1.0, "the residual method has no value boost (use kv with heads='all')"
        span_set = set(dst_pos)
        pin = [q for q in range(min(dst_pos), G["piv_d"]) if q not in span_set]
        c_src = PP.layer_outputs(mt, G["src_tok"], src_pos, LAYERS)
        c_dst = PP.layer_outputs(mt, G["dst_tok"], pin, LAYERS) if pin else None
        pos = list(dst_pos) + pin
        cache = {l: torch.cat([c_src[l], c_dst[l]]) if pin else c_src[l] for l in LAYERS}
        # layer 0 reads k/v from the token EMBEDDINGS (not a layer output), so the span's embeddings are swapped too:
        # without this the pivot's layer-0 attention would still see DST's task tokens (progress tokens are identical)
        src_ids = G["src_tok"]["input_ids"][0, src_pos]

        def stack():
            s = ExitStack()
            s.enter_context(embedding_patch(src_ids, dst_pos))
            s.enter_context(PP.residual_patch(mt, cache, pos, LAYERS))
            return s
        return stack
    raise ValueError(method)


@contextmanager
def embedding_patch(src_ids, dst_pos):
    """Replace the input-embedding output at `dst_pos` with the embeddings of `src_ids` (prompt pass only)."""
    emb = mt._model.get_input_embeddings()
    with torch.no_grad():
        e_src = emb(src_ids.to(emb.weight.device))

    def hook(module, inp, out):
        if out.shape[1] <= max(dst_pos):
            return out
        out = out.clone()
        out[:, dst_pos] = e_src.to(out.dtype)
        return out

    h = emb.register_forward_hook(hook)
    try:
        yield
    finally:
        h.remove()


def run_patched(G, stack, readout=False):
    with stack():
        gen = FUNC.greedy(G["dst_tok"], max_new=MAX_NEW, ctx=PP_CTX)
    ro = None
    if readout:
        def iface(mt_, inputs, patches=None, model_kwargs=None):
            with stack():
                return mt_._model(**dict(inputs), **dict(model_kwargs or {}))
        ro = FUNC.goalscope_readout("", G["dst_tok"], G["piv_d"], heads=READOUT_HEADS, iface=iface,
                                    ctx=PP_CTX)[0]["patched_gen"]
    return gen, ro


def judge(G, clean, gen, ro):
    raw = dict(**clean, gen_patched=gen, ro_patched=ro)
    causal, readout, _ = PP.judge(G["inst"], raw, model_name=ORACLE_MODEL)
    return causal, readout


@torch.no_grad()
def pivot_logits(G, stack):
    with stack():
        return mt._model(input_ids=G["dst_tok"]["input_ids"], attention_mask=G["dst_tok"]["attention_mask"]).logits[0, -1].float()

## Sanity checks

1. **Pre-pivot latents.** `kv`: every position before the pivot must equal the clean DST run (the transfer acts from the
   pivot on). `residual`: the span equals SRC's residuals, every other pre-pivot position equals clean DST.
2. **`kv` (all heads, V×1) ≡ `residual`** at the pivot, per span: max |Δ logit| and the top-1 token (bf16, so expect
   small but nonzero differences).

In [ ]:
pre = list(range(G["piv_d"]))
h_dst = PP.layer_outputs(mt, G["dst_tok"], pre, LAYERS)
t = Table(title="1 · pre-pivot latents vs clean DST (max |Δ| over all layers)")
for c in ("config", "non-span positions", "span positions vs SRC"):
    t.add_column(c)
for m, h, s, v in [("kv", "goal250", "task+progress", 4.0), ("kv", "all", "task+progress", 4.0),
                   ("residual", None, "task", 1.0), ("residual", None, "progress", 1.0), ("residual", None, "task+progress", 1.0)]:
    st = make_stack(G, m, s, h, v)
    with st():
        h_pat = PP.layer_outputs(mt, G["dst_tok"], pre, LAYERS)
    span_d = set(G["W"][s][1])
    keep = [i for i, q in enumerate(pre) if m == "kv" or q not in span_d]
    d_keep = max(float((h_pat[l][keep] - h_dst[l][keep]).abs().max()) for l in LAYERS)
    if m == "residual":
        idx = [pre.index(q) for q in G["W"][s][1]]
        h_src = PP.layer_outputs(mt, G["src_tok"], G["W"][s][0], LAYERS)
        d_span = max(float((h_pat[l][idx] - h_src[l]).abs().max()) for l in LAYERS)
    else:
        d_span = float("nan")
    t.add_row(cfg_name(m, h, s, v), f"{d_keep:.3g}  {'OK' if d_keep == 0 else 'CHANGED'}",
              "—" if m == "kv" else f"{d_span:.3g}  {'OK' if d_span < 1e-2 else 'MISMATCH'}")
Console(width=140).print(t)

t = Table(title="2 · kv (all heads, V×1) vs residual at the pivot")
for c in ("span", "max |Δ logit|", "top-1 kv", "top-1 residual", "top-1 clean DST"):
    t.add_column(c)
top = lambda lg: repr(mt.tokenizer.decode([int(lg.argmax())]))
lg_dst = pivot_logits(G, ExitStack)
for s in ("task", "progress", "task+progress"):
    a, b = pivot_logits(G, make_stack(G, "kv", s, "all", 1.0)), pivot_logits(G, make_stack(G, "residual", s))
    t.add_row(s, f"{float((a - b).abs().max()):.3f}", top(a), top(b), top(lg_dst))
Console(width=140).print(t)

          1 · pre-pivot latents vs clean DST (max |Δ| over all layers)           
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━┓
┃ config                           ┃ non-span positions ┃ span positions vs SRC ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━┩
│ kv·goal250 · task+progress · V×4 │ 0  OK              │ —                     │
│ kv·all · task+progress · V×4     │ 0  OK              │ —                     │
│ residual · task · V×1            │ 0  OK              │ 0  OK                 │
│ residual · progress · V×1        │ 0  OK              │ 0  OK                 │
│ residual · task+progress · V×1   │ 0  OK              │ 0  OK                 │
└──────────────────────────────────┴────────────────────┴───────────────────────┘

## One instance — every config

Clean SRC / DST once (with readouts if `READOUT`), then each config in `CONFIGS`: patched generation, GoalScope at ` =`,
and the oracle verdict (`PP.judge`: c = the source frontier, BOTH is a miss).

In [ ]:
clean = PP.clean_runs(dict(src_tok=G["src_tok"], dst_tok=G["dst_tok"]), readout_heads=READOUT_HEADS, max_new=MAX_NEW,
                      readout=READOUT, ctx=PP_CTX)
rich_print(Panel(Text(clean["gen_src"].strip()), title=f"[red1]clean SRC (c = {inst.src_frontier.id})[/red1]"
                 + (f" · readout: {escape(clean['ro_src'])}" if READOUT else ""), title_align="left"))
rich_print(Panel(Text(clean["gen_dst"].strip()), title=f"[sea_green3]clean DST (d = {inst.dst_frontier.id})[/sea_green3]"
                 + (f" · readout: {escape(clean['ro_dst'])}" if READOUT else ""), title_align="left"))
ONE = []
for m, h, s, v in CONFIGS:
    gen, ro = (clean["gen_dst"], clean.get("ro_dst")) if m == "none" else run_patched(G, make_stack(G, m, s, h, v), READOUT)
    causal, readout = judge(G, clean, gen, ro)
    ONE.append(dict(cfg=cfg_name(m, h, s, v), causal=causal["label"], readout=readout["patched"] if readout else None,
                    gen=gen, ro=ro))
    first = next((ln for ln in gen.split("\n") if ln.strip()), "").strip()
    rich_print(f"[bold]{escape(ONE[-1]['cfg']):<34}[/bold] causal=[bold]{causal['label']:<8}[/bold]"
               + (f" readout={readout['patched']:<8}" if readout else "") + f" · {escape(first[:90])}")

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
You have set `compile_config`, but we are unable to meet the criteria for compilation. Compilation will be skipped.


pos=132 ' =' => I'm calculating the product of all numbers in the list. If the list contains a zero, the result 
will be zero since multiplying any number by zero results in zero.

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

pos=132 ' =' => I'm counting unique elements using `len(set(your_list))`.

╭─ clean SRC (c = product_all) · readout: I'm calculating the product of all numbers in the list. If the list con─╮
│ 1                                                                                                               │
│     for x in nums:                                                                                              │
│         c *= x                                                                                                  │
│     return a, b, c                                                                                              │
│ </code>                                                                                                         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

... (37 outputs left out)

residual · progress · V×1          causal=dst      readout=dst      · len(set(nums))

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

pos=132 ' =' => I'm calculating the product of all numbers in the list.

residual · task+progress · V×1     causal=src      readout=src      · 1

## Sweep — the 2 × 2 grid over instances

Clean runs once per instance (no readouts unless `SWEEP_READOUT`), then every config. Rows are stored in `SWEEP`
(keyed by the config name, so re-running with more instances / configs adds to it).

**Resume** (`RESUME = True`): each instance's clean runs and per-config results are cached in `results/notebook_caches/task_vs_progress/<model>/<datatype>__<src>-<dst>__seed…__mx<MAX_NEW>__ro<0|1>/`, one JSON per instance, written after every config. Re-running skips cached (instance, config) pairs; new configs in `CONFIGS` or new instances run fresh. Delete the folder (or set `RESUME = False`) to recompute.

In [ ]:
SWEEP_IDX = range(32)          # instances of DATATYPE (N_SAMPLES drawn)
SWEEP_READOUT = False
RESUME = False                  # reuse cached clean runs + config results per instance (results/notebook_caches/...)
import hashlib
CACHE_DIR = os.path.join(env_utils.DEFAULT_RESULTS_DIR, "notebook_caches", "task_vs_progress", mt.name.split("/")[-1],
                         f"{PP._safe(DATATYPE)}__{SRC_LANG}-{DST_LANG}__seed{SEED}__mx{MAX_NEW}__ro{int(SWEEP_READOUT)}")
os.makedirs(CACHE_DIR, exist_ok=True)


def cache_path(inst):
    ik = PP.inst_key(inst)
    return os.path.join(CACHE_DIR, f"{inst.src_frontier.id}__{inst.dst_frontier.id}__{hashlib.md5(ik.encode()).hexdigest()[:10]}.json")


def cache_write(fp, entry):
    tmp = fp + ".tmp"
    with open(tmp, "w") as f:
        json.dump(entry, f, indent=1)
    os.replace(tmp, fp)


SWEEP = globals().get("SWEEP", collections.defaultdict(list))
n_run = n_hit = 0
for i in SWEEP_IDX:
    inst_i = INSTS[i]
    fp = cache_path(inst_i)
    entry = json.load(open(fp)) if (RESUME and os.path.exists(fp)) else {}
    if entry.get("key") not in (None, PP.inst_key(inst_i)):
        entry = {}                                             # hash collision / stale file: recompute
    entry.update(key=PP.inst_key(inst_i), idx=i)
    entry.setdefault("configs", {})
    Gi = ci = None
    line = []
    for m, h, s, v in CONFIGS:
        name = cfg_name(m, h, s, v)
        row = entry["configs"].get(name)
        if row is None:                                        # not cached: run it (geometry + clean runs only once)
            if Gi is None:
                Gi = geometry(inst_i)
                ci = entry.get("clean")
                if ci is None:
                    ci = PP.clean_runs(dict(src_tok=Gi["src_tok"], dst_tok=Gi["dst_tok"]), readout_heads=READOUT_HEADS,
                                       max_new=MAX_NEW, readout=SWEEP_READOUT, ctx=PP_CTX)
                    entry["clean"] = ci
            gen, ro = (ci["gen_dst"], ci.get("ro_dst")) if m == "none" else run_patched(Gi, make_stack(Gi, m, s, h, v), SWEEP_READOUT)
            causal, readout = judge(Gi, ci, gen, ro)
            row = dict(idx=i, method=m, heads=h, span=s, v=v, causal=causal["label"], shifted=causal["label"] == "src",
                       readout=readout["patched"] if readout else None, ro_shifted=readout["shifted"] if readout else None,
                       gen=gen, ro=ro, c=inst_i.src_frontier.id, d=inst_i.dst_frontier.id)
            entry["configs"][name] = row
            cache_write(fp, entry)                             # after every config, so an interrupt loses at most one
            n_run += 1
        else:
            n_hit += 1
        key = (DATATYPE, name)
        SWEEP[key] = [r for r in SWEEP[key] if r["idx"] != i] + [dict(row, idx=i)]
        line.append("■" if row["causal"] == "src" else "·")
    print(f"[{i:2d}] {inst_i.src_frontier.id} vs {inst_i.dst_frontier.id}: {' '.join(line)}")
print(f"configs run: {n_run} · from cache: {n_hit} · cache: {CACHE_DIR}")

[ 0] product_all vs distinct_count: · · ■ · ■ ■ ■ ■ · · ■ ■ · ■
[ 1] value_range vs count_positive: · · ■ · ■ ■ ■ ■ ■ ■ ■ ■ ■ ■
[ 2] is_sorted vs all_positive: · ■ ■ · · ■ ■ ■ · · ■ ■ · ■
[ 3] count_even vs all_positive: · ■ ■ · · ■ ■ ■ · · ■ ■ · ■
[ 4] count_negative vs mean_int: · · ■ · ■ ■ · ■ ■ ■ ■ · ■ ■
[ 5] min_val vs has_even: · · · · ■ ■ · · · ■ ■ · · ■
[ 6] max_val vs contains_zero: · · ■ · · · ■ ■ ■ ■ ■ ■ ■ ■
[ 7] product_all vs all_positive: · · · · · · · · · · ■ · · ■
[ 8] contains_zero vs count_even: · ■ ■ · · ■ ■ ■ · · ■ ■ · ■
[ 9] product_all vs count_even: · ■ ■ · · ■ ■ ■ · · ■ ■ · ■
[10] count_negative vs has_even: · ■ ■ · · ■ ■ ■ · · ■ ■ · ■
[11] product_all vs contains_zero: · · · · · · · · · · ■ · · ■
[12] all_positive vs distinct_count: · ■ ■ · · ■ ■ ■ · · ■ ■ · ■
[13] is_sorted vs count_negative: · ■ ■ · ■ ■ ■ ■ ■ ■ ■ ■ ■ ■
[14] min_val vs mean_int: · · ■ · · ■ ■ ■ · · ■ ■ · ■
[15] distinct_count vs contains_zero: · · ■ · · ■ · ■ · ■ ■ · · ■
[16] all_positive vs c

## Results — 2 × 2 grids and the full config table

Causal-shift (patched writes c) per config; each 2 × 2 grid fixes the method / heads / V and varies which span the pivot
reads from SRC (the no-op corner is the clean DST run).

In [ ]:
def rate(rs, f="shifted"):
    xs = [r[f] for r in rs if r[f] is not None]
    return (sum(xs) / len(xs)) if xs else float("nan")

for dt in dict.fromkeys(k[0] for k in SWEEP):
    t = Table(title=f"{dt} · all configs", show_lines=False)
    for c, j in (("config", "left"), ("n", "right"), ("causal-shift", "right"), ("readout-shift", "right"), ("verdicts", "left")):
        t.add_column(c, justify=j)
    for (d, name), rs in SWEEP.items():
        if d != dt:
            continue
        t.add_row(name, str(len(rs)), f"{rate(rs):.2f}", f"{rate(rs, 'ro_shifted'):.2f}",
                  " ".join(f"{k}:{n}" for k, n in collections.Counter(r["causal"] for r in rs).most_common()))
    Console(width=140).print(t)

    noop = SWEEP.get((dt, "clean DST (no-op)"), [])
    groups = dict.fromkeys((r["method"], r["heads"], r["v"]) for (d, _), rs in SWEEP.items() if d == dt for r in rs[:1]
                           if r["method"] != "none")
    for m, h, v in groups:
        cell = {}
        for s in ("task", "progress", "task+progress"):
            cell[s] = SWEEP.get((dt, cfg_name(m, h, s, v)))
        g2 = Table(title=f"{dt} · {m}{'·' + h if h else ''} · V×{v:g} — causal-shift", show_lines=True)
        g2.add_column(""); g2.add_column("progress from DST", justify="center"); g2.add_column("progress from SRC", justify="center")
        f = lambda rs: f"{rate(rs):.2f} (n={len(rs)})" if rs else "—"
        g2.add_row("task from DST", f(noop), f(cell["progress"]))
        g2.add_row("task from SRC", f(cell["task"]), f(cell["task+progress"]))
        Console(width=100).print(g2)

                                           list · all configs                                            
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ config                           ┃  n ┃ causal-shift ┃ readout-shift ┃ verdicts                       ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ clean DST (no-op)                │ 32 │         0.00 │           nan │ dst:32                         │
│ kv·goal250 · task · V×1          │ 32 │         0.38 │           nan │ dst:19 src:12 neither:1        │
│ kv·goal250 · task · V×4          │ 32 │         0.81 │           nan │ src:26 neither:4 dst:2         │
│ kv·goal250 · progress · V×1      │ 32 │         0.00 │           nan │ dst:32                         │
│ kv·goal250 · progress · V×4      │ 32 │         0.28 │           nan │ dst:18 src:9 neither:4 both:1  │
│ kv·goal250 · task+progress · V×1 │ 32 │         0.81 │           nan │ src:26 dst:5 neither:1         │
│ kv·all · task · V×1              │ 32 │         0.66 │           nan │ src:21 dst:7 neither:4         │
│ kv·all · task · V×4              │ 32 │         0.91 │           nan │ src:29 dst:2 neither:1         │
│ kv·all · progress · V×1          │ 32 │         0.16 │           nan │ dst:26 src:5 neither:1         │
│ kv·all · progress · V×4          │ 32 │         0.38 │           nan │ dst:17 src:12 neither:2 both:1 │
│ kv·all · task+progress · V×1     │ 32 │         0.97 │           nan │ src:31 neither:1               │
│ residual · task · V×1            │ 32 │         0.66 │           nan │ src:21 dst:7 neither:4         │
│ residual · progress · V×1        │ 32 │         0.16 │           nan │ dst:26 src:5 neither:1         │
│ residual · task+progress · V×1   │ 32 │         0.97 │           nan │ src:31 neither:1               │
└──────────────────────────────────┴────┴──────────────┴───────────────┴────────────────────────────────┘

         list · kv·goal250 · V×1 — causal-shift          
┏━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃               ┃ progress from DST ┃ progress from SRC ┃
┡━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ task from DST │    0.00 (n=32)    │    0.00 (n=32)    │
├───────────────┼───────────────────┼───────────────────┤
│ task from SRC │    0.38 (n=32)    │    0.81 (n=32)    │
└───────────────┴───────────────────┴───────────────────┘

         list · kv·goal250 · V×4 — causal-shift          
┏━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃               ┃ progress from DST ┃ progress from SRC ┃
┡━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ task from DST │    0.00 (n=32)    │    0.28 (n=32)    │
├───────────────┼───────────────────┼───────────────────┤
│ task from SRC │    0.81 (n=32)    │         —         │
└───────────────┴───────────────────┴───────────────────┘

           list · kv·all · V×1 — causal-shift            
┏━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃               ┃ progress from DST ┃ progress from SRC ┃
┡━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ task from DST │    0.00 (n=32)    │    0.16 (n=32)    │
├───────────────┼───────────────────┼───────────────────┤
│ task from SRC │    0.66 (n=32)    │    0.97 (n=32)    │
└───────────────┴───────────────────┴───────────────────┘

           list · kv·all · V×4 — causal-shift            
┏━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃               ┃ progress from DST ┃ progress from SRC ┃
┡━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ task from DST │    0.00 (n=32)    │    0.38 (n=32)    │
├───────────────┼───────────────────┼───────────────────┤
│ task from SRC │    0.91 (n=32)    │         —         │
└───────────────┴───────────────────┴───────────────────┘

          list · residual · V×1 — causal-shift           
┏━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃               ┃ progress from DST ┃ progress from SRC ┃
┡━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ task from DST │    0.00 (n=32)    │    0.16 (n=32)    │
├───────────────┼───────────────────┼───────────────────┤
│ task from SRC │    0.66 (n=32)    │    0.97 (n=32)    │
└───────────────┴───────────────────┴───────────────────┘